In [5]:
import argparse

import numpy as np
import pandas as pd
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from pycox.evaluation import EvalSurv


def calculate_c_index_with_ci(df, target_time=1.0, n_iterations=1000):
    y_true = df['label'].values
    y_surv_prob = (1 - df['pos_probs'].values)
    durations_test = np.full(len(y_true), target_time)

    # A. 计算全量数据集上的结果 (作为 Mean)
    surv_full = pd.DataFrame(y_surv_prob.reshape(1, -1), index=[target_time])
    ev_full = EvalSurv(surv_full, durations_test, y_true.astype(np.int64), censor_surv='km')
    full_score = ev_full.concordance_td()

    # B. Bootstrap 仅用于计算 CI
    stats = []
    print(f"正在通过 Bootstrap 计算 C-index 的置信区间...")
    for i in range(n_iterations):
        indices = resample(range(len(y_true)), n_samples=len(y_true), random_state=i)
        sub_y_true = y_true[indices].astype(np.int64)
        if len(np.unique(sub_y_true)) < 2: continue
        
        sub_y_prob = y_surv_prob[indices]
        sub_durations = durations_test[indices]
        
        surv_df = pd.DataFrame(sub_y_prob.reshape(1, -1), index=[target_time])
        ev = EvalSurv(surv_df, sub_durations, sub_y_true, censor_surv='km')
        stats.append(ev.concordance_td())

    lower = np.percentile(stats, 2.5)
    upper = np.percentile(stats, 97.5)
    return full_score, lower, upper

# ---------------------------------------------------------
# 2. 分类指标计算 (均值取自全量数据集)
# ---------------------------------------------------------
def calculate_classification_metrics(df, n_iterations=1000):
    y_true = df['label'].values
    y_scores = df['pos_probs'].values
    y_pred = df['prediction'].values

    # A. 直接计算完整测试集的结果
    raw_results = {
        'auc': roc_auc_score(y_true, y_scores),
        'acc': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'pre': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0)
    }

  
    boot_stats = {'auc': [], 'acc': [], 'f1': [], 'pre': [], 'recall': []}
    print(f"正在通过 Bootstrap 计算分类指标的置信区间...")
    for i in range(n_iterations):
        indices = resample(range(len(y_true)), n_samples=len(y_true), random_state=i)
        sub_true = y_true[indices]
        if len(np.unique(sub_true)) < 2: continue

        sub_scores = y_scores[indices]
        sub_pred = y_pred[indices]

        boot_stats['auc'].append(roc_auc_score(sub_true, sub_scores))
        boot_stats['acc'].append(accuracy_score(sub_true, sub_pred))
        boot_stats['f1'].append(f1_score(sub_true, sub_pred, zero_division=0))
        boot_stats['pre'].append(precision_score(sub_true, sub_pred, zero_division=0))
        boot_stats['recall'].append(recall_score(sub_true, sub_pred, zero_division=0))

    final_results = {}
    for key, mean_val in raw_results.items():
        lower = np.percentile(boot_stats[key], 2.5)
        upper = np.percentile(boot_stats[key], 97.5)
        final_results[key] = (mean_val, lower, upper)

    return final_results

In [6]:
import pandas as pd

# 读取CSV
df = pd.read_csv('/home/user/prognosis_lst/根据结果直接计算指标/data/dense_huaxi.csv')

# 重命名为标准格式（与test函数返回的格式一致）
df = df.rename(columns={
    'pred': 'pos_probs',
    'label': 'label'
})

# 添加prediction列（根据阈值0.5将概率转为类别）
df['prediction'] = (df['pos_probs'] >= 0.5).astype(int)

# 只保留需要的三列
df = df[['label', 'prediction', 'pos_probs']]

c_val, c_low, c_high = calculate_c_index_with_ci(df)
clf_metrics = calculate_classification_metrics(df)

# -----------------------------------------------------
# 输出报表 (这里的 Mean 是整个测试集的真实得分)
# -----------------------------------------------------
print("\n" + "="*60)
print(f"{'指标 (Full Test Set)':<20} | {'结果 (Score)':<10} | {'95% CI (Bootstrap)':<20}")
print("-" * 60)

print(f"{'C-index (pycox)':<20} | {c_val:.4f}     [{c_low:.4f} - {c_high:.4f}]")

display_map = {
    'auc': 'AUC (ROC)', 'acc': 'Accuracy', 'f1': 'F1-Score',
    'pre': 'Precision', 'recall': 'Recall'
}


for key, name in display_map.items():
    score, low, high = clf_metrics[key]
    print(f"{name:<20} | {score:.4f}     [{low:.4f} - {high:.4f}]")
    

正在通过 Bootstrap 计算 C-index 的置信区间...
正在通过 Bootstrap 计算分类指标的置信区间...

指标 (Full Test Set)   | 结果 (Score) | 95% CI (Bootstrap)  
------------------------------------------------------------
C-index (pycox)      | 0.5682     [0.4818 - 0.6574]
AUC (ROC)            | 0.5942     [0.4669 - 0.7135]
Accuracy             | 0.6024     [0.4940 - 0.6988]
F1-Score             | 0.6118     [0.4800 - 0.7191]
Precision            | 0.5532     [0.4107 - 0.6863]
Recall               | 0.6842     [0.5385 - 0.8286]


In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc

# 读取CSV
df = pd.read_csv('/home/user/prognosis_lst/根据结果直接计算指标/data/m3d_huaxi.csv')

# 重命名为标准格式（与test函数返回的格式一致）
df = df.rename(columns={
    'pred': 'pos_probs',
    'label': 'label'
})

df_yue = pd.read_csv("/home/user/prognosis_lst/根据结果直接计算指标/data/m3d_train.csv")
# =====================================================
# 1. 基于约登指数 (Youden's J) 计算最佳 cutoff
# =====================================================
fpr, tpr, thresholds = roc_curve(df['label'], df['pos_probs'])
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_cutoff = thresholds[best_idx]

print(f"\n最佳阈值 (Youden‘s J): {best_cutoff:.4f}")
print(f"对应 Sensitivity: {tpr[best_idx]:.4f}, Specificity: {1-fpr[best_idx]:.4f}")

# 用约登阈值生成预测标签
df['prediction'] = (df['pos_probs'] >= best_cutoff).astype(int)


df = df[['label', 'prediction', 'pos_probs']].copy()

c_val, c_low, c_high = calculate_c_index_with_ci(df)
clf_metrics_original = calculate_classification_metrics(df)



# =====================================================
# 4. 输出报表对比
# =====================================================
print("\n" + "="*70)
print(f"{'指标':<20} | '阈值:{best_cutoff}':<25 | {'约登最佳阈值':<25}")
print("-" * 70)

# C-index
print(f"{'C-index (pycox)':<20} | {c_val:.4f} [{c_low:.4f}-{c_high:.4f}]  ")

display_map = {
    'auc': 'AUC (ROC)', 'acc': 'Accuracy', 'f1': 'F1-Score',
    'pre': 'Precision', 'recall': 'Recall'
}

for key, name in display_map.items():
    score_orig, low_orig, high_orig = clf_metrics_original[key]
    print(f"{name:<20} | {score_orig:.4f} [{low_orig:.4f}-{high_orig:.4f}] ")

# 额外输出最佳阈值本身
print("-" * 70)
print(f"{'最佳阈值':<20} | {0.5:<25} | {best_cutoff:.4f}")
print("="*70)


最佳阈值 (Youden‘s J): 0.2471
对应 Sensitivity: 0.6957, Specificity: 0.7091
正在通过 Bootstrap 计算 C-index 的置信区间...
正在通过 Bootstrap 计算分类指标的置信区间...

指标                   | '阈值:0.24707031':<25 | 约登最佳阈值                   
----------------------------------------------------------------------
C-index (pycox)      | 0.6460 [0.5541-0.7485]  
AUC (ROC)            | 0.6555 [0.5375-0.7659] 
Accuracy             | 0.7030 [0.6139-0.7921] 
F1-Score             | 0.6809 [0.5625-0.7843] 
Precision            | 0.6667 [0.5293-0.8040] 
Recall               | 0.6957 [0.5714-0.8270] 
----------------------------------------------------------------------
最佳阈值                 | 0.5                       | 0.2471
